# Frozen v2 mechanistic rerun

This notebook uses only the strict benchmark-bound v2 runner. It does not use the legacy mutable-eval-set pipeline.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os
import subprocess

REPO_URL = 'https://github.com/urosavurdic/dpo-safety-representations.git'
REPO_DIR = '/content/dpo-safety-representations'
BRANCH = 'agent/c-quadrant-end-to-end-e0e2317a'
PINNED_COMMIT = 'REPLACE_AFTER_PUSH_WITH_COMMIT_SHA'

if not os.path.exists(REPO_DIR):
    subprocess.run([
        'git', 'clone', '-b', BRANCH,
        REPO_URL, REPO_DIR,
    ], check=True)

os.chdir(REPO_DIR)
subprocess.run(['git', 'fetch', 'origin'], check=True)
subprocess.run([
    'git', 'checkout', BRANCH,
], check=True)
subprocess.run([
    'git', 'pull', '--ff-only', 'origin', BRANCH,
], check=True)

commit = subprocess.check_output([
    'git', 'rev-parse', 'HEAD',
], text=True).strip()
assert PINNED_COMMIT != 'REPLACE_AFTER_PUSH_WITH_COMMIT_SHA', 'Set PINNED_COMMIT after pushing the patch.'
assert commit == PINNED_COMMIT, f'Wrong commit: {commit}'
print('Checked out exact commit:', commit)


In [ ]:
!python -m pip install -q -r requirements.txt
!python -m pip uninstall -y torchao || true
!python -m compileall src
!pytest tests/ -q


In [ ]:
import json
import subprocess

latest = json.load(open(
    'data/frozen_v2/LATEST_BENCHMARK.json'
))
bench = latest['benchmark_path']

subprocess.run([
    'python', '-m',
    'src.create_direction_split_manifest',
    '--benchmark', bench,
], check=True)

subprocess.run([
    'python', '-m',
    'src.validate_benchmark_v2',
    '--benchmark', bench,
    '--review-csv',
    'data/review/c_review_queue.csv',
    '--gate-config',
    'logs/benchmark_gate_config.json',
    '--split-manifest',
    'logs/direction_split_manifest.json',
], check=True)

status = json.load(open(
    'logs/benchmark_validation_status.json'
))
static_fields = [
    'schema_integrity_pass',
    'prompt_integrity_pass',
    'c_review_pass',
    'c_review_mapping_pass',
    'benchmark_hash_pass',
    'split_benchmark_hash_pass',
    'split_hash_pass',
]
assert all(status.get(k) is True for k in static_fields), status
print('Static benchmark checks passed.')
print('Artifact freshness:', status['artifact_freshness_pass'])


In [ ]:
!bash rerun_mechanistic_v2.sh --dry-run --regenerate --with-probes


In [ ]:
RUN_GPU = False
if RUN_GPU:
    !bash rerun_mechanistic_v2.sh --regenerate --with-probes


In [ ]:
if RUN_GPU:
    import json
    import subprocess
    latest = json.load(open(
        'data/frozen_v2/LATEST_BENCHMARK.json'
    ))
    subprocess.run([
        'python', '-m',
        'src.validate_benchmark_v2',
        '--benchmark', latest['benchmark_path'],
        '--review-csv',
        'data/review/c_review_queue.csv',
        '--gate-config',
        'logs/benchmark_gate_config.json',
        '--split-manifest',
        'logs/direction_split_manifest.json',
    ], check=True)
    status = json.load(open(
        'logs/benchmark_validation_status.json'
    ))
    assert status['technical_benchmark_status'] == 'PASS', status
    print('Fresh v2 validation passed.')
